# Simulação de opinião pública sobre desigualdade no Brasil com LLMs

Projeto final de IA 2026/1, Mackenzie. **Bruno Ferrão (RA 10401081)** — prof. Rogério de Oliveira.

Peguei a pesquisa CESOP/IPEC 04839 sobre como o brasileiro enxerga desigualdade (2.000 entrevistas presenciais, julho de 2023) e usei o Qwen 2.5 0.5B Instruct pra simular as respostas de uma amostra de 200 pessoas em duas afirmações Likert. Em seguida treinei um Random Forest com as mesmas variáveis pra ter um termo de comparação.

O notebook está organizado em três blocos: preparação dos dados, simulação com LLM, e baseline supervisionado. Roda em CPU em uns 10 a 15 minutos, ou em GPU T4 do Colab gratuito em cerca de 3.


## Instalação e imports

Se for primeira execução em Colab ou ambiente novo, descomente a linha do pip.

In [ ]:
# !pip install -q pandas pyreadstat scikit-learn matplotlib seaborn transformers torch

import os, sys, time, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, mean_absolute_error, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from scipy.stats import entropy

SEED = 42
N_SAMPLE = 200
N_REPS = 3
TEMPERATURE = 0.8
MAX_NEW_TOKENS = 6
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

DATA_PATH = Path("../data/raw/04839.sav")
OUT_DIR = Path("..")
FIGS_DIR = OUT_DIR / "figs"; FIGS_DIR.mkdir(exist_ok=True)
DATA_DIR = OUT_DIR / "data"; DATA_DIR.mkdir(exist_ok=True)

np.random.seed(SEED); torch.manual_seed(SEED)


## Dados

O arquivo SPSS original (`.sav`) tem 49 colunas. A maioria são respostas a perguntas individuais codificadas como inteiros, com tabelas de label separadas no metadado. Pra trabalhar com isso confortavelmente, decodifico em português os códigos das variáveis sociodemográficas que vou usar como preditoras, e mantenho as duas variáveis-alvo (P6A e P6C) como inteiros de 1 a 5.

In [ ]:
df_raw, meta = pyreadstat.read_sav(str(DATA_PATH))
print(f"Pesquisa: {df_raw.shape[0]} respondentes, {df_raw.shape[1]} colunas")

SEXO_MAP = {1: "Masculino", 2: "Feminino"}
RACA_MAP = {1: "Branca", 2: "Preta", 3: "Parda", 4: "Amarela", 5: "Indigena"}
RELIGIAO_MAP = {
    1:"Catolica", 2:"Assembleia de Deus", 3:"Batista/Metodista/Presbiteriana",
    4:"Universal do Reino de Deus", 5:"Deus e Amor", 6:"Evangelho Quadrangular",
    7:"Igreja Internacional da Graca", 8:"Renascer em Cristo", 9:"Sara Nossa Terra",
    10:"Outras Evangelicas", 11:"Evangelica nao especificada", 12:"Adventista",
    13:"Testemunha de Jeova", 14:"Judaica", 15:"Espirita/Kardecista",
    16:"Afro-Brasileira", 17:"Oriental", 18:"Outras religioes",
    19:"Religioso sem religiao especifica/Agnostico", 20:"Ateu, sem religiao"
}
REND_MAP = {1:"Mais de 20 SM", 2:"10 a 20 SM", 3:"5 a 10 SM", 4:"2 a 5 SM",
            5:"1 a 2 SM", 6:"Ate 1 SM", 98:"Sem rendimento", 99:"Nao respondeu"}
REGIAO_MAP = {1:"Norte", 2:"Nordeste", 3:"Sudeste", 4:"Sul", 5:"Centro-Oeste"}

def faixa_idade(i):
    if pd.isna(i): return None
    i = int(i)
    if i <= 17: return "16-17"
    if i <= 24: return "18-24"
    if i <= 34: return "25-34"
    if i <= 44: return "35-44"
    if i <= 59: return "45-59"
    return "60+"

def escolaridade_grupo(e):
    if pd.isna(e): return None
    e = int(e)
    if e <= 10: return "Fundamental incompleto"
    if e == 11: return "Fundamental completo"
    if e in (12, 13): return "Medio incompleto"
    if e == 14: return "Medio completo"
    if e == 15: return "Superior incompleto"
    if e == 16: return "Superior completo"

df = pd.DataFrame({
    "sexo": df_raw["SEXO"].map(SEXO_MAP),
    "idade": df_raw["IDADE"],
    "faixa_idade": df_raw["IDADE"].apply(faixa_idade),
    "escolaridade": df_raw["ESCOLARIDADE"].apply(escolaridade_grupo),
    "raca_cor": df_raw["RACA_COR"].map(RACA_MAP),
    "religiao": df_raw["RELIGIÃO"].map(RELIGIAO_MAP),
    "renda_familiar": df_raw["REND2"].map(REND_MAP),
    "regiao": df_raw["REGIAO"].map(REGIAO_MAP),
    "P6A": df_raw["P6A"],
    "P6C": df_raw["P6C"],
})
df.head()


Limpeza: tiro respondentes com dados faltantes em qualquer das preditoras ou nas duas perguntas-alvo, e descarto também respostas "Não sabe" e "Não respondeu" (códigos 98 e 99) das variáveis Likert. Sobram 1.854 pessoas dos 2.000 originais.

In [ ]:
required = ["sexo","faixa_idade","escolaridade","raca_cor","religiao",
            "renda_familiar","regiao","P6A","P6C"]
df_clean = df.dropna(subset=required).copy()
df_clean = df_clean[df_clean["P6A"].isin([1,2,3,4,5])]
df_clean = df_clean[df_clean["P6C"].isin([1,2,3,4,5])]
df_clean["P6A"] = df_clean["P6A"].astype(int)
df_clean["P6C"] = df_clean["P6C"].astype(int)
print(f"Apos limpeza: {len(df_clean)} respondentes")

print("\nDistribuicao real das respostas:")
for q in ["P6A", "P6C"]:
    counts = df_clean[q].value_counts(normalize=True).sort_index().round(3)
    print(f"  {q}: {dict(counts)}")


## Amostragem

Sorteio 200 respondentes com estratificação por sexo × região, pra garantir cobertura mínima de cada cruzamento e respeitar o requisito do projeto de simular pelo menos 10% dos dados.

In [ ]:
df_clean["_strat"] = df_clean["sexo"] + "_" + df_clean["regiao"]
sample = df_clean.groupby("_strat", group_keys=False).apply(
    lambda x: x.sample(min(len(x), max(1, int(N_SAMPLE * len(x) / len(df_clean)))),
                       random_state=SEED)
).reset_index(drop=True)

if len(sample) > N_SAMPLE:
    sample = sample.sample(n=N_SAMPLE, random_state=SEED).reset_index(drop=True)
elif len(sample) < N_SAMPLE:
    extra = df_clean[~df_clean.index.isin(sample.index)].sample(
        N_SAMPLE - len(sample), random_state=SEED)
    sample = pd.concat([sample, extra]).reset_index(drop=True)

sample = sample.drop(columns=["_strat"])
sample.to_csv(DATA_DIR / "sample_200.csv", index=False)
print(f"Amostra: {len(sample)} respondentes")
sample.head()


## Carregando o LLM

Escolhi o **Qwen 2.5 0.5B Instruct** por três motivos: licença Apache 2.0 (totalmente aberto, atende a regra do projeto), suporte nativo a português e chat template, e tamanho que cabe em qualquer notebook (menos de 1 GB em FP32). É um modelo pequeno de propósito, pra testar o piso de capacidade de LLMs nessa tarefa sem depender de GPU ou de API paga.

In [ ]:
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.requires_grad_(False)
print(f"Modelo carregado em {time.time()-t0:.1f}s")


## Engenharia de prompt

O prompt tem duas partes. Uma instrução de sistema que fixa o papel do modelo (respondente da pesquisa) e o formato de saída (um número de 1 a 5, sem texto). Uma mensagem de usuário que lista o perfil sociodemográfico e a afirmação a ser respondida.

In [ ]:
SYSTEM_PROMPT = '''Voce sera apresentado a um perfil sociodemografico de um respondente brasileiro de uma pesquisa de opiniao publica do IPEC/Cesop sobre desigualdade no Brasil (2023). Sua tarefa e simular qual seria a resposta MAIS PROVAVEL desse respondente a uma afirmacao, considerando os atributos do perfil e padroes de opiniao tipicos no Brasil.

Responda SOMENTE com um numero entre 1 e 5, conforme a escala:
1 = Concorda totalmente
2 = Concorda em parte
3 = Nao concorda, nem discorda
4 = Discorda em parte
5 = Discorda totalmente

Nao explique. Nao escreva texto. Apenas o numero.'''

QUESTIONS = {
    "P6A": "A abordagem policial e baseada no tipo de cabelo, tipo de vestimenta e cor de pele das pessoas.",
    "P6C": "Aumentar a representatividade de pessoas negras, mulheres e populacao LGBTQIA+ na politica e em cargos de poder contribui para diminuir as desigualdades estruturais.",
}

def build_prompt(row, qtext):
    user = f'''Perfil do respondente:
- Sexo: {row['sexo']}
- Faixa etaria: {row['faixa_idade']}
- Escolaridade: {row['escolaridade']}
- Cor/Raca: {row['raca_cor']}
- Religiao: {row['religiao']}
- Renda familiar: {row['renda_familiar']}
- Regiao do pais: {row['regiao']}

Afirmacao: "{qtext}"

Qual a resposta MAIS PROVAVEL desse respondente?
Responda apenas com o numero (1, 2, 3, 4 ou 5):'''
    msgs = [{"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse_answer(text):
    m = re.search(r'[1-5]', text)
    return int(m.group(0)) if m else None

print(build_prompt(sample.iloc[0], QUESTIONS["P6A"]))


## Loop de simulação

Cada respondente da amostra é apresentado ao modelo três vezes pra cada uma das duas questões, com sampling estocástico (`temperature=0.8`, `top_p=0.9`). No total, 1.200 inferências. Em CPU Intel típica leva cerca de 10 minutos. Em GPU cai pra 2 ou 3.

In [ ]:
results = []
t0 = time.time()
total = len(sample) * len(QUESTIONS) * N_REPS
done = 0

for rep in range(N_REPS):
    for qid, qtext in QUESTIONS.items():
        for idx, row in sample.iterrows():
            prompt = build_prompt(row, qtext)
            inputs = tok(prompt, return_tensors="pt")
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=TEMPERATURE,
                    top_p=0.9,
                    pad_token_id=tok.eos_token_id,
                )
            gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            results.append({
                "rep": rep, "question": qid, "resp_idx": idx,
                "real": row[qid], "pred_raw": gen.strip(),
                "pred": parse_answer(gen),
            })
            done += 1
            if done % 100 == 0:
                el = time.time() - t0
                eta = (el/done) * (total - done)
                print(f"  [{done}/{total}] {el:.0f}s decorridos | eta {eta:.0f}s")

res_df = pd.DataFrame(results)
res_df.to_csv(DATA_DIR / "llm_predictions.csv", index=False)
print(f"\nFinalizado em {time.time()-t0:.0f}s. Predicoes nulas: {res_df['pred'].isna().sum()} de {len(res_df)}")


## Métricas do LLM

Acurácia (exact match na Likert), MAE (erro médio em pontos da escala) e divergência KL entre a distribuição predita e a real. Onde o regex não conseguiu extrair um número 1–5, imputo a moda da amostra real — não quero descartar a linha porque distorce a distribuição.

In [ ]:
for qid in QUESTIONS:
    mode_val = int(sample[qid].mode().iloc[0])
    mask = (res_df["question"] == qid) & res_df["pred"].isna()
    res_df.loc[mask, "pred"] = mode_val
res_df["pred"] = res_df["pred"].astype(int)

llm_metrics = {}
for qid in QUESTIONS:
    sub = res_df[res_df["question"] == qid]
    real = sub["real"].astype(int)
    pred = sub["pred"]
    acc = accuracy_score(real, pred)
    mae = mean_absolute_error(real, pred)
    real_dist = real.value_counts(normalize=True).reindex([1,2,3,4,5], fill_value=1e-9).values
    pred_dist = pred.value_counts(normalize=True).reindex([1,2,3,4,5], fill_value=1e-9).values
    kl = entropy(pred_dist, real_dist)
    llm_metrics[qid] = {"accuracy": acc, "mae": mae, "kl": kl}
    print(f"{qid}:  acc={acc:.3f}   mae={mae:.3f}   kl={kl:.3f}")
    print(f"  real: {dict(zip([1,2,3,4,5], np.round(real_dist,3)))}")
    print(f"  llm:  {dict(zip([1,2,3,4,5], np.round(pred_dist,3)))}")
    print()


## Baseline supervisionado: Random Forest

Random Forest com 200 árvores e profundidade 8, validação cruzada estratificada em 5 folds. Codifico as 7 preditoras com Label Encoding (suficiente pra um RF). Em seguida treino um modelo no conjunto inteiro pra extrair as importâncias relativas das variáveis — esses números entram na análise de explicabilidade.

In [ ]:
features = ["sexo","faixa_idade","escolaridade","raca_cor",
            "religiao","renda_familiar","regiao"]

df_feat = sample.copy()
for f in features:
    df_feat[f] = LabelEncoder().fit_transform(df_feat[f].astype(str))

rf_metrics = {}
feat_importances = {}
rf_preds_byq = {}

for qid in QUESTIONS:
    X = df_feat[features].values
    y = df_feat[qid].astype(int).values

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    accs, maes = [], []
    all_preds = np.zeros_like(y)
    for tr, te in skf.split(X, y):
        clf = RandomForestClassifier(n_estimators=200, max_depth=8,
                                     random_state=SEED, n_jobs=-1)
        clf.fit(X[tr], y[tr])
        pred = clf.predict(X[te])
        all_preds[te] = pred
        accs.append(accuracy_score(y[te], pred))
        maes.append(mean_absolute_error(y[te], pred))
    rf_preds_byq[qid] = all_preds

    clf_full = RandomForestClassifier(n_estimators=200, max_depth=8,
                                       random_state=SEED, n_jobs=-1)
    clf_full.fit(X, y)
    feat_importances[qid] = dict(zip(features, clf_full.feature_importances_))

    rf_metrics[qid] = {
        "acc_mean": np.mean(accs), "acc_std": np.std(accs),
        "mae_mean": np.mean(maes),
    }
    print(f"{qid}:  acc={np.mean(accs):.3f}±{np.std(accs):.3f}   mae={np.mean(maes):.3f}")
    top = sorted(feat_importances[qid].items(), key=lambda x: -x[1])
    print("  importancias:", [(k, round(v,3)) for k,v in top])
    print()


## Visualizações

Quatro figuras: distribuições comparativas, matrizes de confusão do LLM, importância das variáveis no RF, e acurácia individual lado a lado.

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, qid in zip(axes, QUESTIONS):
    sub = res_df[res_df["question"] == qid]
    real = sub["real"].astype(int).value_counts(normalize=True).reindex([1,2,3,4,5], fill_value=0)
    llm = sub["pred"].value_counts(normalize=True).reindex([1,2,3,4,5], fill_value=0)
    rf = pd.Series(rf_preds_byq[qid]).value_counts(normalize=True).reindex([1,2,3,4,5], fill_value=0)
    xpos = np.arange(5); w = 0.27
    ax.bar(xpos-w, real.values, w, label="Real", color="#2E86AB")
    ax.bar(xpos, llm.values, w, label="LLM (Qwen2.5)", color="#E63946")
    ax.bar(xpos+w, rf.values, w, label="Random Forest", color="#F4A261")
    ax.set_xticks(xpos)
    ax.set_xticklabels(["Conc.\nTotal","Conc.\nParte","Neutro","Disc.\nParte","Disc.\nTotal"], fontsize=9)
    ax.set_ylabel("Frequencia relativa")
    ax.set_title(f"{qid}: {QUESTIONS[qid][:60]}...")
    ax.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR/"01_distribuicoes.png", dpi=120)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, qid in zip(axes, QUESTIONS):
    sub = res_df[res_df["question"] == qid]
    cm = confusion_matrix(sub["real"].astype(int), sub["pred"], labels=[1,2,3,4,5])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", ax=ax,
                xticklabels=["1","2","3","4","5"], yticklabels=["1","2","3","4","5"])
    ax.set_xlabel("LLM")
    ax.set_ylabel("Real")
    ax.set_title(f"Confusion Matrix - {qid}")
plt.tight_layout()
plt.savefig(FIGS_DIR/"02_confusion_llm.png", dpi=120)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, qid in zip(axes, QUESTIONS):
    items = sorted(feat_importances[qid].items(), key=lambda x: x[1])
    ax.barh([i[0] for i in items], [i[1] for i in items], color="#264653")
    ax.set_xlabel("Importancia")
    ax.set_title(f"Feature Importance (RF) - {qid}")
plt.tight_layout()
plt.savefig(FIGS_DIR/"03_feature_importance.png", dpi=120)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
qids = list(QUESTIONS.keys())
llm_acc = [llm_metrics[q]["accuracy"] for q in qids]
rf_acc = [rf_metrics[q]["acc_mean"] for q in qids]
xpos = np.arange(len(qids)); w = 0.35
ax.bar(xpos-w/2, llm_acc, w, label="LLM", color="#E63946")
ax.bar(xpos+w/2, rf_acc, w, label="Random Forest", color="#F4A261")
ax.set_xticks(xpos); ax.set_xticklabels(qids)
ax.set_ylabel("Acuracia")
ax.set_title("LLM vs Random Forest - Acuracia individual")
ax.set_ylim(0, max(max(llm_acc), max(rf_acc)) * 1.3)
for i, (l, r) in enumerate(zip(llm_acc, rf_acc)):
    ax.text(i-w/2, l+0.01, f"{l:.2f}", ha="center")
    ax.text(i+w/2, r+0.01, f"{r:.2f}", ha="center")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR/"04_accuracy_comparison.png", dpi=120)
plt.show()


## Discussão dos resultados

O Random Forest venceu em todas as métricas. Acurácia de 0,335 (P6A) e 0,440 (P6C), contra 0,145 e 0,130 do LLM. KL quatro a seis vezes menor.

O achado mais interessante não é a vitória do RF, é o **viés do LLM**. Olhando o gráfico de distribuições: a população real concentra suas respostas em "Concorda totalmente" (39% em P6A, 45% em P6C). O LLM faz o oposto — 51% e 67% das suas predições caem em "Discorda totalmente", e ele literalmente não predita "Concorda totalmente" em nenhuma das duas questões. Zero.

A hipótese mais plausível é alinhamento defensivo. Modelos *instruction-tuned* são otimizados pra evitar afirmações fortes em temas socialmente sensíveis. Diante de "a polícia discrimina por cor", a resposta "segura" é discordar. Outras duas hipóteses não devem ser descartadas: capacidade insuficiente (500M parâmetros contra os 13-70B usuais na literatura) e cobertura de treino pobre em português brasileiro.

O Random Forest, por sua vez, atinge boa acurácia jogando seguro — prevê a moda da amostra (68% e 73% em "Concorda totalmente"). Acerta a direção, mas perde a variância da população.

Religião e faixa etária dominam a feature importance, com 17 a 20% cada. Sexo é a variável menos informativa, com 7 a 8%. Isso pode surpreender em afirmações sobre raça e gênero, mas é consistente com pesquisas que mostram idade, religião e escolaridade como divisores mais fortes da opinião brasileira do que o gênero do respondente.
